#This first part is only to conect to kaggle and Bring a dataset in google colab

In [ ]:
from google.colab import files

files.upload()  # This will prompt you to upload a kaggle token file from the local machine.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
%%capture
!pip install kaggle

In [7]:
# Download latest version
!kaggle datasets download -d divyanshu2000/doctor-healthcare-100k

Dataset URL: https://www.kaggle.com/datasets/divyanshu2000/doctor-healthcare-100k
License(s): apache-2.0




  0%|          | 0.00/41.8M [00:00<?, ?B/s]
100%|██████████| 41.8M/41.8M [00:00<00:00, 2.01GB/s]


# #This first part is only to conect to kaggle and Bring a dataset in vs code

In [6]:
import shutil
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    src = Path('kaggle.json')
    if src.exists():
        shutil.copy(src, kaggle_json)
        kaggle_json.chmod(0o600)
        print(f"Copied kaggle.json to {kaggle_json}")
    else:
        print(f"ERROR: Place your kaggle.json here: {kaggle_json}")
else:
    print(f"Credentials already at: {kaggle_json}")

Credentials already at: C:\Users\stron\.kaggle\kaggle.json


In [9]:
!unzip doctor-healthcare-100k.zip

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
import kaggle
from pathlib import Path

notebook_dir = Path().resolve()  # or hardcode: Path(r'C:\Users\stron\Desktop\github\LLM')

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'divyanshu2000/doctor-healthcare-100k',
    path=str(notebook_dir),
    unzip=True
)
print(f"Saved to: {notebook_dir}")



Dataset URL: https://www.kaggle.com/datasets/divyanshu2000/doctor-healthcare-100k
Saved to: C:\Users\stron\Desktop\github\LLM


In [1]:
import pandas as pd
df = pd.read_csv('doctor-healthcare-100k/Doctor-HealthCare-100k.csv')
df.head()
print(len(df))

112156


#Bringing all the required libraries

In [3]:
%%capture
# this line hides all the download and install logs for installing all these libraries
%pip install -U transformers
%pip install -U datasets
%pip install -U accelerate
%pip install -U peft
%pip install -U trl            # Transformer Reinforcement Learning and Supervised Fine Tuning
%pip install -U bitsandbytes     # for quantization 4bit and 8 bit for QLoRA
%pip install -U wandb

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

modelName = "google/gemma-2-2b-it"

bnbConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # safer than bfloat16 on many consumer GPUs
)

tokenizer = AutoTokenizer.from_pretrained(modelName)

model = AutoModelForCausalLM.from_pretrained(
    modelName,
    device_map="auto",
    quantization_config=bnbConfig
)

Loading weights: 100%|██████████| 288/288 [00:01<00:00, 199.50it/s]


In [4]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.5.1+cu121
True
12.1


In [2]:
input_text = "How are you today? how have you been recently?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=200)
print(outputs)
print(tokenizer.decode(outputs[0]))

tensor([[     2,   2299,    708,    692,   3646, 235336,   1368,    791,    692,
           1125,   7378, 235336,    109, 235285, 235303, 235262,   3900,   1578,
         235269,   7593,    692, 235341,    590, 235303,    524,   1125,  13572,
            675,   1160,    578,   1009,   3749,   7340, 235269,    901,   8691,
         235269,    590, 235303, 235262,   4915, 235265,   2250,   1105,    692,
         235336,  44416, 235248,    108,    107]], device='cuda:0')
<bos>How are you today? how have you been recently?

I'm doing well, thank you! I've been busy with work and some personal projects, but overall, I'm happy. How about you? 😊 
<end_of_turn>


# Fine-tuning Steps for Gemma 2 Using LoRA On top of Qlora 4 bit Quantizattion

In [3]:
import os
#!pip uninstall -y datasets pyarrow
#!pip install pyarrow datasets

from transformers import (
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)

In [4]:
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)


In [5]:
import wandb

In [6]:
from datasets import load_dataset

: 

In [9]:
from trl import SFTTrainer

In [11]:
run = wandb.init(
    project='Fine-tune Gemma-2-2b-it on doctor-healthcare dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


#Loading Model and Tokeniuzers

In [13]:
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

QLORA config

In [15]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)


Load Model

In [18]:
base_model = "google/gemma-2-2b-it"
dataset_name = df
new_model = "Gemma-2-2b-it"

In [19]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
model

In [ ]:
print(model.named_modules())

The function traverses the quantized model, identifies all Linear4bit layers ( all the layer that has been quantized when the model was loaded load_in_4bit=True) created by BitsAndBytes, extracts their module names, and returns them as LoRA target modules. This allows PEFT to automatically inject LoRA adapters into the model's attention and feed-forward projection layers without manually specifying each layer


some people actually hard code that as modules = ['down_proj', 'o_proj', 'up_proj', 'v_proj', 'gate_proj', 'q_proj', 'k_proj']


In [20]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [ ]:
print(modules)

In [ ]:
df.columns

In [ ]:
df.head(2)

In [21]:
def format_chatml(example):
    text = (
        "<|im_start|>system\n"
        f"{example['instruction']}"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['input']}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}"
        "<|im_end|>"
    )

    return {"text": text}

In [22]:
from datasets import Dataset # Import the Dataset class

# Convert pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Now map the format_example function to the Hugging Face Dataset
dataset = hf_dataset.map(format_chatml)

Map:   0%|          | 0/112156 [00:00<?, ? examples/s]

In [ ]:
dataset

In [ ]:
dataset['text'][1]


<|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
Hi i am a teenager. about 2 days a go i found about 5 slim lumps across my forehead. do you no what this could be my mum says that it is just boils but im worried could help me. also i have been have a lot of headaches/migraines as well. it also herts when i touch them.<|im_end|>
<|im_start|>assistant
Hi, Dear I studied your query in all it details and I understood your concerns. Cause - On whatever limited facts given you seem to have Acne, or pimples, and they are painful to touch. The migraine or headaches is a separate ailment and don't correlate with painful acne on forehead. So don't worry.Hence, To reduce your worry Please consult for opinion from ER doctor. Plz hit thanks and write excellent Reviews if this would resolve your query. Plz don't worry and do Welcome for any further query in this regard to me. Have a Good Day. Chat Doctor. N.<|im_end|>

In [23]:
# LoRA config LoRA adds small, low-rank matrices to each layer, allowing only these matrices to be trained.
#This minimizes the computational load and memory needed.
tokenizer.chat_template = None  # this disables any built-in chat formatting
peft_config = LoraConfig(
    r=16,     #  the rank in LoRA directly affects the number of trainable parameters in the model. more rank more parameters to train
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",  # dont train biases, only train lora adapters
    task_type="CAUSAL_LM",
    target_modules=modules,
)
model = get_peft_model(model, peft_config)

In [ ]:
model

small inference

In [26]:
dataset = dataset.train_test_split(test_size=0.1)
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 100940
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 11216
    })
})

#Training

In [ ]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    report_to="wandb"
)

# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_arguments,
)

model.config.use_cache = False
trainer.train()

Adding EOS to train dataset:   0%|          | 0/100940 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100940 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/11216 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11216 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss
